In [ ]:
!pip install pandas numpy scipy scikit-learn geopandas rasterio pyproj shapely folium tqdm chardet requests
# pykma-grid가 필요한 경우 별도 설치: !pip install pykma-grid

In [1]:
import os
os.makedirs("./output", exist_ok=True)
print("output 폴더 준비 완료")

output 폴더 준비 완료


### 1. 데이터 전처리

#### 1.1 병합 데이터

In [2]:
import pandas as pd
import chardet

def detect_encoding(file_path):
    with open(file_path, 'rb') as f:
        result = chardet.detect(f.read())
    return result['encoding']

file_paths = [
    "./input/풍력발전량(2020.3분기).csv",
    "./input/풍력발전량(2020.4분기).csv",
    "./input/풍력발전량(2021.1분기).csv",
    "./input/풍력발전량(2021.2분기).csv"
]

dataframes = []
for file in file_paths:
    encoding = detect_encoding(file)
    dataframes.append(pd.read_csv(file, encoding=encoding))

jeju = pd.concat(dataframes, ignore_index=True)

jeju_1year = pd.melt(
    jeju,
    id_vars=["일자"],
    var_name="단지",
    value_name="발전량"
)

jeju_1year = jeju_1year.sort_values(by=["일자", "단지"]).reset_index(drop=True)
jeju_1year = jeju_1year[jeju_1year["단지"] != "전체"].reset_index(drop=True)

#### 1.2 위경도, 설비용량, 갯수

In [3]:
LaLo = pd.read_csv("./input/한국에너지공단_풍력기 위치정보_20221231.csv", encoding="cp949")

LaLo.columns = ["단지번호", "단지명", "풍력기번호(임시)", "준공일", "분류", "주소", "GRS80_C_x", "GRS80_C_y", "위도", "경도"]

lalo_simplified = LaLo[["단지명", "위도", "경도"]]

lalo_simplified["단지명"] = lalo_simplified["단지명"].replace({"가시리": "가시", "동복북촌": "동복"})

# '단지'와 '단지명'을 기준으로 데이터 병합
jeju_1year = jeju_1year.merge(lalo_simplified, left_on="단지", right_on="단지명", how="left")

jeju_1year.drop(columns=["단지명"], inplace=True)

# 추가 정보를 포함한 딕셔너리 생성
additional_info = {
    "단지": ["가시", "동복", "행원", "김녕", "신창"],
    "발전용량": [15000, 30000, 7880, 750, 1700],
    "발전기대수": [13, 15, 7, 1, 2]
}

additional_info_df = pd.DataFrame(additional_info)

# '단지'를 기준으로 추가 정보를 jeju_1year 데이터프레임에 병합
jeju_1year = jeju_1year.merge(additional_info_df, on="단지", how="left")

C:\Users\MI2RL\AppData\Local\Temp\ipykernel_11584\3692042324.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lalo_simplified["단지명"] = lalo_simplified["단지명"].replace({"가시리": "가시", "동복북촌": "동복"})


#### 1.3 고도

In [ ]:
import json
import requests
from tqdm import tqdm

google_api_key = "YOUR_GOOGLE_API_KEY"

coord = jeju_1year[['위도','경도']]
Elevation = pd.DataFrame({'Elevation':[],'Latitude':[],'Longitude':[]})

for i in tqdm(range(len(coord))):
    lat, lon = float(coord.loc[i,'위도']), float(coord.loc[i,'경도'])
    url = f"https://maps.googleapis.com/maps/api/elevation/json?locations={lat},{lon}&key={google_api_key}"
    payload = {}
    headers = {}
    response = requests.request("GET", url, headers=headers, data=payload)

    response_json = json.loads(response.text)

    # 새로운 행 구성: 고도, 위도, 경도 데이터 추가
    new_row = {
        'Elevation': response_json['results'][0]['elevation'],
        'Latitude': response_json['results'][0]['location']['lat'],
        'Longitude': response_json['results'][0]['location']['lng']
    }

    jeju_1year = pd.concat([jeju_1year, pd.DataFrame([new_row])], ignore_index=True)

In [4]:
jeju_1year['고도'] = pd.read_csv('./input/고도.csv',encoding='cp949')['Elevation']

#### 1.4 경사도

In [5]:
from pyproj import Proj, transform
from scipy.spatial import KDTree

seo = pd.read_csv("./input/서귀포경사도.csv")
seo = seo.dropna()

jeju = pd.read_csv("./input/제주경사도.csv")
jeju = jeju.dropna()

# 두 데이터프레임에서 경사도 관련 열 이름을 통일
seo = seo.rename(columns={'slope1': 'slope'})
jeju = jeju.rename(columns={'slope11': 'slope'})

merged_df = pd.merge(seo, jeju, on=['Longitude', 'Latitude'], how='outer')

# 병합된 경사도 열(slope_x, slope_y)을 통합하여 새로운 slope 열로 생성 후 불필요한 열 제거
merged_df['slope'] = merged_df['slope_x'].fillna(merged_df['slope_y'])
merged_df.drop(columns=['slope_x', 'slope_y'], inplace=True)

merged_df = merged_df.rename(columns={'Longitude': '경도', 'Latitude': '위도'})

tree = KDTree(merged_df[['위도', '경도']])

def find_nearest_slope(row):
    if pd.isna(row['위도']) or pd.isna(row['경도']):
        return float('nan')
    dist, idx = tree.query([row['위도'], row['경도']])
    return merged_df['slope'].iloc[idx]

jeju_1year['경사도'] = jeju_1year.apply(find_nearest_slope, axis=1)

#### 1.5 풍속, 풍향 일관성

In [ ]:
from scipy.spatial import cKDTree
import numpy as np

wind = pd.read_csv('./input/wind.csv',encoding='cp949')

wind['일시'] = wind['일시'].apply(lambda x: x.replace(".", "-"))

# 날짜 형식 변경 및 열 이름 변경
wind.rename(columns={'일시':'일자'},inplace=True)
date = wind[['지점명','일자']]
values = wind[['평균 풍속(m/s)', '풍향(deg)', '위도', '경도']]

# 결측치 채움: 평균 값으로 대체
values.fillna(values.mean(),inplace=True)

# 날짜와 측정 지점명을 기준으로 그룹화
concated = pd.concat([date,values],axis=1)
concated.groupby(['지점명','일자']).count()

# 날짜를 분리하여 '년', '월' 열 추가
jeju_1year['년'] = jeju_1year['일자'].apply(lambda x: x.split('-')[0])
jeju_1year['월'] = jeju_1year['일자'].apply(lambda x: x.split('-')[1])
concated['년'] = concated['일자'].apply(lambda x: x.split('-')[0])
concated['월'] = concated['일자'].apply(lambda x: x.split('-')[1])

# 풍속 데이터의 좌표 정보를 KDTree로 공간 검색 준비
wind_coords = concated[["년","월","위도", "경도"]]
wind_tree = cKDTree(wind_coords)

# jeju_1year의 각 좌표와 가장 가까운 wind 데이터 좌표 찾기
background_coords = jeju_1year[["년","월","위도", "경도"]]
distances, indices = wind_tree.query(background_coords, k=1)

# 가장 가까운 풍속과 풍향 데이터를 가져옴
query_result = pd.DataFrame(indices)
query_result.rename(columns={0:'index'},inplace=True)
result_df = pd.DataFrame(concated.loc[query_result.loc[:,'index'],['평균 풍속(m/s)','풍향(deg)']])

jeju_1year["wind_speed"] = pd.DataFrame({"windspeed":np.array(result_df['평균 풍속(m/s)']),
                                  "deg":np.array(result_df['풍향(deg)'])})['windspeed']
jeju_1year["wind_direction"] = pd.DataFrame({"windspeed":np.array(result_df['평균 풍속(m/s)']),
                                  "deg":np.array(result_df['풍향(deg)'])})['deg']

# 풍향 일관성 함수
def direction_consistency(wind_direction):
    radian = np.radians(wind_direction)

    x_components = np.cos(radian)
    y_components = np.sin(radian)

    mean_x = np.mean(x_components)
    mean_y = np.mean(y_components)

    magnitude = np.sqrt(mean_x**2 + mean_y**2)

    return magnitude

# '년', '월' 기준으로 풍향 일관성 계산
jeju_1year['풍향 일관성'] = jeju_1year.groupby(['년', '월'])['wind_direction'].transform(lambda x: direction_consistency(x))

jeju_1year.drop(['년','월'],axis=1,inplace=True)
jeju_1year.rename(columns={'wind_speed':'풍속','wind_direction':'풍향'},inplace=True)
jeju_1year.drop(columns=['풍향'], inplace=True)

#### 1.6 접근 도로와의 거리

In [ ]:
from scipy.spatial import KDTree

roads_data = pd.read_csv("./input/LaLo_roads_street.csv", encoding="cp949")

jeju_coords = jeju_1year[['위도', '경도']].to_numpy()
roads_coords = roads_data[['위도', '경도']].to_numpy()

roads_tree = KDTree(roads_coords)

distances, _ = roads_tree.query(jeju_coords)

jeju_1year['도로와의 거리'] = distances

jeju_1year.to_csv("./input/jeju_1year.csv", index=False)

### 2. 주요 데이터와의 상관관계 분석을 통한 우선순위 도출

In [8]:
jeju_1year = pd.read_csv("./input/jeju_1year.csv")

jeju_1year['1대당 발전량'] = jeju_1year['발전량'] / jeju_1year['발전기대수']
jeju_1year['1대당 발전용량'] = jeju_1year['발전용량'] / jeju_1year['발전기대수']
jeju_1year['균일한 발전량'] = jeju_1year['1대당 발전량'] / jeju_1year['1대당 발전용량']

correlation_features = ['고도', '경사도', '풍속', '풍향 일관성', '도로와의 거리']

corr_result = jeju_1year[['균일한 발전량'] + correlation_features].corr()['균일한 발전량'].drop('균일한 발전량').sort_values(ascending=False)

corr_result.to_csv("./output/correlation.csv", header=True)
print("저장: output/correlation.csv")
corr_result

저장: output/correlation.csv


풍속         0.126370
풍향 일관성    -0.009053
도로와의 거리   -0.055402
경사도       -0.113614
고도        -0.132509
Name: 균일한 발전량, dtype: float64

### 3. AHP를 활용한 가중치 설정

In [9]:
import numpy as np
import pandas as pd

pairwise_matrix = np.array([
    [1,    1.5,  3,    2,    1.5],   # 풍향 일관성
    [1/1.5, 1,    2.5,  1.5,  1.25], # 풍속
    [1/3,   1/2.5, 1,    1.5,  2],    # 도로와의 거리
    [1/2,   1/1.5, 1/1.5, 1,    1.5], # 경사도
    [1/1.5, 1/1.25, 1/2,  1/1.5, 1],  # 고도
])

col_sums = pairwise_matrix.sum(axis=0)

normalized_matrix = pairwise_matrix / col_sums

priority_vector = normalized_matrix.mean(axis=1)

criteria = ['풍속', '풍향 일관성', '도로와의 거리', '경사도', '고도']
results = pd.DataFrame({
    "Criterioaan": criteria,
    "Priority Weight": priority_vector,
})

In [10]:
# 4. λ_max 계산
weighted_sum = np.dot(pairwise_matrix, priority_vector)
lambda_max = np.mean(weighted_sum / priority_vector)

# 5. CI (Consistency Index) 계산
n = pairwise_matrix.shape[0]  # 기준의 개수
ci = (lambda_max - n) / (n - 1)

# 6. CR (Consistency Ratio) 계산
# RI 값은 기준의 개수에 따라 정해짐
random_index = {1: 0.0, 2: 0.0, 3: 0.58, 4: 0.90, 5: 1.12, 6: 1.24, 7: 1.32}
ri = random_index[n]
cr = ci / ri if ri != 0 else 0

print(f"lambda_max : {lambda_max:.4f}")
print(f"CI         : {ci:.4f}")
print(f"RI         : {ri:.4f}")
print(f"CR         : {cr:.4f}")

results

lambda_max : 5.2613
CI         : 0.0653
RI         : 1.1200
CR         : 0.0583


,Criterioaan,Priority Weight
0,풍속,0.311500
1,풍향 일관성,0.232607
2,도로와의 거리,0.165633
3,경사도,0.150884
4,고도,0.139376


In [11]:
# AHP 결과 저장
import pandas as pd
pd.DataFrame({
    "metric": ["lambda_max", "CI", "RI", "CR"],
    "value": [lambda_max, ci, ri, cr]
}).to_csv("./output/ahp_consistency.csv", index=False)

results.to_csv("./output/ahp_weights.csv", index=False)
print("저장: output/ahp_consistency.csv")
print("저장: output/ahp_weights.csv")

저장: output/ahp_consistency.csv
저장: output/ahp_weights.csv


### 4. 100m 단위의 그리드 데이터 전처리

In [14]:
import geopandas as gpd
from pykma_grid.coordinate import Coordinate
import pandas as pd
import numpy as np
from shapely.geometry import Point
from shapely.prepared import prep
from tqdm import tqdm

def create_grid(geometry, spacing=100):
    bounds = geometry.bounds

    # 지정된 간격(spacing)으로 경계 범위 내 격자점 생성
    x_range = np.arange(bounds[0], bounds[2], spacing)
    y_range = np.arange(bounds[1], bounds[3], spacing)
    
    # 격자점 생성기 (Point 객체 반환)
    for x in x_range:
        for y in y_range:
            yield Point(x, y)

def process_region(file_path):
    gdf = gpd.read_file(file_path)
    
    if gdf.crs is not None:
        gdf = gdf.to_crs("EPSG:5179")
    else:
        gdf = gdf.set_crs("EPSG:5179")
    
    # 격자점 생성
    grid_points = []
    for _, row in tqdm(gdf.iterrows(), total=len(gdf), desc=f"Processing {file_path}"):
        prepared_geom = prep(row.geometry)
        for point in create_grid(row.geometry):
            if prepared_geom.contains(point):
                grid_points.append(point)
    
    # GeoDataFrame 생성 및 좌표계 변환
    grid_gdf = gpd.GeoDataFrame(geometry=grid_points, crs="EPSG:5179")
    grid_gdf = grid_gdf.to_crs("EPSG:4326")
    
    return pd.DataFrame({
        '경도': grid_gdf.geometry.x,
        '위도': grid_gdf.geometry.y
    })

if __name__ == "__main__":
    jeju_grid_df = process_region("./input/제주도/제주도 제주시.shp")
    seoguipo_grid_df = process_region("./input/제주도/제주도 서귀포시.shp")
    
    jeju_grid = pd.concat([jeju_grid_df, seoguipo_grid_df], ignore_index=True)
    
    jeju_grid.to_csv("./input/jeju_grid.csv", 
                           index=False, 
                           encoding='utf-8-sig',
                           float_format='%.7f')

Processing ./input/제주도/제주도 서귀포시.shp: 100%|██████████| 1/1 [00:05<00:00,  5.07s/it]


#### 1. 보호구역 제외

In [15]:
import numpy as np
import pandas as pd
import rasterio
from pyproj import Transformer
import folium
from tqdm import tqdm
from folium.plugins import FastMarkerCluster

# 제주도 좌표 범위 설정
JEJU_BOUNDS = {
    'lat_min': 33.195,  # 제주도 최저 위도
    'lat_max': 33.566,  # 제주도 최고 위도
    'lon_min': 126.160, # 제주도 최저 경도
    'lon_max': 126.947  # 제주도 최고 경도
}

# 좌표 변환기 초기화
def initialize_transformer(src_crs="EPSG:4326", dst_crs="EPSG:5179"):
    return Transformer.from_crs(src_crs, dst_crs, always_xy=True)

# 위도와 경도를 변환하여 평면 좌표로 반환
def transform_coords(lon, lat, transformer):
    x, y = transformer.transform(lon, lat)
    return x, y

def process_coordinates_in_chunks(file_path, transformer, chunk_size=10000):
    with open(file_path, 'r') as file:
        total_rows = sum(1 for line in file) - 1
    
    valid_coords_list = []
    
    for chunk in tqdm(pd.read_csv(file_path, chunksize=chunk_size),
                     total=total_rows//chunk_size + 1,
                     desc="processing Jeju region"):
        # 제주도 영역 필터링
        jeju_mask = (
            (chunk['Latitude'] >= JEJU_BOUNDS['lat_min']) &
            (chunk['Latitude'] <= JEJU_BOUNDS['lat_max']) &
            (chunk['Longitude'] >= JEJU_BOUNDS['lon_min']) &
            (chunk['Longitude'] <= JEJU_BOUNDS['lon_max'])
        )
        chunk = chunk[jeju_mask]
        
        if len(chunk) > 0:
            vfunc = np.vectorize(lambda x, y: transform_coords(x, y, transformer))
            x_coords, y_coords = vfunc(chunk['Longitude'].values, chunk['Latitude'].values)
            
            chunk['Projected_X'] = x_coords
            chunk['Projected_Y'] = y_coords
            valid_coords_list.append(chunk)
            # 각 파일 청크에서 유효한 좌표만 필터링하고 변환된 좌표 추가
    
    return pd.concat(valid_coords_list) if valid_coords_list else pd.DataFrame()

# 주어진 좌표가 보호구역 래스터 내에 있는지 확인
def check_protected_areas_vectorized(x_coords, y_coords, raster, transform):
    rows, cols = rasterio.transform.rowcol(transform, x_coords, y_coords)
    mask = (rows >= 0) & (rows < raster.shape[0]) & (cols >= 0) & (cols < raster.shape[1])
    result = np.zeros_like(rows, dtype=bool)
    result[mask] = raster[rows[mask], cols[mask]] != 0
    return result

# 필터링된 좌표를 클러스터 형태로 지도에 표시
def create_clustered_map(filtered_coords):
    center_lat = (JEJU_BOUNDS['lat_min'] + JEJU_BOUNDS['lat_max']) / 2
    center_lon = (JEJU_BOUNDS['lon_min'] + JEJU_BOUNDS['lon_max']) / 2
    
    m = folium.Map(location=[center_lat, center_lon], zoom_start=10)
    
    points = filtered_coords[['Latitude', 'Longitude']].values.tolist()
    FastMarkerCluster(points).add_to(m)
    
    return m

if __name__ == "__main__":
    with rasterio.open("./input/보호구역래스터.tif") as src:
        raster = src.read(1)
        transform = src.transform
        transformer = initialize_transformer("EPSG:4326", src.crs.to_string())
    
    coordinates_df = process_coordinates_in_chunks("./input/grid_points.csv", transformer)
    
    if not coordinates_df.empty:
        protected = check_protected_areas_vectorized(
            coordinates_df['Projected_X'].values,
            coordinates_df['Projected_Y'].values,
            raster,
            transform
        )
        
        # 보호구역에 포함된 좌표 필터링
        filtered_coords = coordinates_df[protected]
        
        output_coords = filtered_coords[['Longitude', 'Latitude']]
        output_coords.to_csv("./제주보호구역.csv", 
                            index=False, 
                            encoding='utf-8-sig',
                            float_format='%.7f')

output_coords.rename(columns={'Latitude':'위도','Longitude':'경도'},inplace=True)

c:\Users\MI2RL\miniconda3\envs\main\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.2) or chardet (7.2.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
processing Jeju region:  81%|████████  | 811/1001 [00:05<00:01, 152.61it/s]C:\Users\MI2RL\AppData\Local\Temp\ipykernel_11584\795962575.py:48: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chunk['Projected_X'] = x_coords
C:\Users\MI2RL\AppData\Local\Temp\ipykernel_11584\795962575.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/index

#### 2. 고도

In [16]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
from pyproj import Transformer
from scipy.spatial import cKDTree
from tqdm import tqdm

file_path = './input/N3L_F0010000_A/N3L_F0010000_A.shp'
contours = gpd.read_file(file_path)

coords = output_coords

# 좌표 변환기 초기화 (WGS84 -> ITRF_2000_UTM_K)
transformer = Transformer.from_proj(
    proj_from="epsg:4326",  # WGS84
    proj_to=contours.crs.to_string(),
    always_xy=True
)

# 좌표 데이터를 GeoDataFrame으로 변환
coords['geometry'] = coords.apply(
    lambda row: Point(transformer.transform(row['경도'], row['위도'])), axis=1
)
jeju_grid = gpd.GeoDataFrame(coords, geometry='geometry', crs=contours.crs)

contour_points = contours.geometry.apply(lambda geom: geom.centroid.coords[0])
contour_points = list(contour_points)
contour_elevations = contours['CONT'].values

kdtree = cKDTree(contour_points)

tqdm.pandas()

# Elevation 추출 함수 정의 (선형 보간법 적용)
def get_elevation_interpolated(point, kdtree, contour_points, contour_elevations):
    """KDTree를 사용해 주어진 점에 가장 가까운 두 개의 등고선을 기반으로 고도를 선형 보간"""
    distances, indices = kdtree.query([point.x, point.y], k=2)
    if len(distances) < 2 or distances[1] == 0:
        return contour_elevations[indices[0]]
    else:
        # 두 등고선 간의 거리 비율로 보간
        z1, z2 = contour_elevations[indices[0]], contour_elevations[indices[1]]
        d1, d2 = distances
        return (z1 * d2 + z2 * d1) / (d1 + d2)

jeju_grid['고도'] = jeju_grid['geometry'].progress_apply(
    lambda point: get_elevation_interpolated(point, kdtree, contour_points, contour_elevations)
)

jeju_grid.drop(columns=['geometry'], inplace=True)

output_file_path = './input/jeju_grid.csv'
jeju_grid[['위도', '경도', '고도']].to_csv(output_file_path, index=False, encoding='utf-8')

C:\Users\MI2RL\AppData\Local\Temp\ipykernel_11584\930305742.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  coords['geometry'] = coords.apply(
100%|██████████| 164866/164866 [00:07<00:00, 21458.33it/s]


#### 3. 경사도

In [17]:
import numpy as np
from pyproj import Proj, transform
from scipy.spatial import KDTree

seo = pd.read_csv("./input/서귀포경사도.csv")
seo = seo.dropna()

jeju = pd.read_csv("./input/제주경사도.csv")
jeju = jeju.dropna()

seo = seo.rename(columns={'slope1': 'slope'})
jeju = jeju.rename(columns={'slope11': 'slope'})

# 위도와 경도를 기준으로 두 데이터프레임을 외부 조인
merged_df = pd.merge(seo, jeju, on=['Longitude', 'Latitude'], how='outer')

merged_df['slope'] = merged_df['slope_x'].fillna(merged_df['slope_y'])
merged_df.drop(columns=['slope_x', 'slope_y'], inplace=True)

merged_df = merged_df.rename(columns={'Longitude': '경도', 'Latitude': '위도'})

tree = KDTree(merged_df[['위도', '경도']])

# 가장 가까운 좌표를 KDTree로 검색해 해당 경사도 반환
def find_nearest_slope(row):
    dist, idx = tree.query([row['위도'], row['경도']])
    return merged_df['slope'].iloc[idx]

jeju_grid.rename(columns={'Latitude': '위도', 'Longitude': '경도'}, inplace=True)
jeju_grid['경사도'] = jeju_grid.apply(find_nearest_slope, axis=1)

jeju_grid.reset_index(drop=True, inplace=True)

#### 4. 풍속, 풍향 일관성

In [18]:
import pandas as pd

wind = pd.read_csv('./input/wind.csv',encoding='cp949')

# 날짜 형식 변경 및 열 이름 수정
wind['일시'] = wind['일시'].apply(lambda x: x.replace(".", "-"))
wind.rename(columns={'일시':'일자'},inplace=True)

# 평균 풍속, 풍향 등 결측치를 열 평균으로 대체
date = wind[['지점명','일자']]
values = wind[['평균 풍속(m/s)', '풍향(deg)', '위도', '경도']]
values.fillna(values.mean(),inplace=True)
concated = pd.concat([date,values],axis=1)

concated.groupby(['지점명','일자']).count()

concated['년'] = concated['일자'].apply(lambda x: x.split('-')[0])
concated['월'] = concated['일자'].apply(lambda x: x.split('-')[1])

# 풍향 일관성을 계산하는 함수
def direction_consistency(wind_direction):
    radian = np.radians(wind_direction)

    x_components = np.cos(radian)
    y_components = np.sin(radian)

    mean_x = np.mean(x_components)
    mean_y = np.mean(y_components)

    magnitude = np.sqrt(mean_x**2 + mean_y**2)

    return magnitude

# 년-월 기준으로 풍향 일관성 계산
concated['년'] = concated['일자'].apply(lambda x: x.split('-')[0])
concated['월'] = concated['일자'].apply(lambda x: x.split('-')[1])
concated['풍향일관성'] = concated.groupby(['년', '월'])['풍향(deg)'].transform(lambda x: direction_consistency(x))

from scipy.spatial import cKDTree

# 풍속 데이터를 cKDTree로 공간 검색, 제주도 그리드와 가장 가까운 풍속 좌표 찾기
wind_coords = concated[["위도", "경도"]]
wind_tree = cKDTree(wind_coords)
background_coords = jeju_grid[["위도", "경도"]]
distances, indices = wind_tree.query(background_coords, k=1)

query_result = pd.DataFrame(indices)
query_result.rename(columns={0:'index'},inplace=True)

# 검색된 인덱스를 통해 관련 풍속 데이터를 추출
result_df = pd.DataFrame(concated.loc[query_result.loc[:,'index'],['풍향일관성','평균 풍속(m/s)','풍향(deg)','년','월']])

jeju_grid["풍속"] = pd.DataFrame({"windspeed":np.array(result_df['평균 풍속(m/s)'])})['windspeed']
jeju_grid["풍향일관성"] = pd.DataFrame({"풍향일관성":np.array(result_df['풍향일관성'])})['풍향일관성']

C:\Users\MI2RL\AppData\Local\Temp\ipykernel_11584\3620595800.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  values.fillna(values.mean(),inplace=True)


#### 5. 접근 도로와의 거리

In [19]:
from scipy.spatial import KDTree

roads_data = pd.read_csv("./input/LaLo_roads_street.csv", encoding="cp949")

jeju_coords = jeju_grid[['위도', '경도']].to_numpy()
roads_coords = roads_data[['위도', '경도']].to_numpy()

# 도로 좌표로 KDTree 생성
roads_tree = KDTree(roads_coords)

# 각 제주도 그리드 좌표와 가장 가까운 도로 좌표 간의 거리 계산
distances, _ = roads_tree.query(jeju_coords)

jeju_grid['도로와의 거리'] = distances

### 5. 점수 산정

In [20]:
from sklearn.preprocessing import MinMaxScaler

columns_to_normalize = ['고도', '경사도', '풍속', '풍향일관성', '도로와의 거리']

scaler = MinMaxScaler()

jeju_grid[columns_to_normalize] = scaler.fit_transform(jeju_grid[columns_to_normalize])

weights = {
    '풍속': 0.311500,
    '풍향일관성': 0.232607,
    '도로와의 거리': 0.165633,
    '경사도': 0.150884,
    '고도': 0.139376
}

jeju_grid['score'] = (
    jeju_grid['풍속'] * weights['풍속'] +
    jeju_grid['풍향일관성'] * weights['풍향일관성'] +
    (1-jeju_grid['도로와의 거리']) * weights['도로와의 거리'] +
    (1-jeju_grid['경사도']) * weights['경사도'] +
    (1-jeju_grid['고도']) * weights['고도']
)

jeju_grid

# 격자 점수 저장
jeju_grid.to_csv("./output/jeju_grid_scored.csv", index=False)
print("저장: output/jeju_grid_scored.csv")

저장: output/jeju_grid_scored.csv


### 6. 높은 score 지도 시각화

In [21]:
turbin = pd.read_csv('./input/한국에너지공단_풍력기 위치정보_20221231.csv', encoding='cp949')
turbin.head()

jeju = turbin[turbin['주소'].str.split().str[0] == '제주특별자치도']
jeju.head()

mean_locations = jeju.groupby('단지명')[['위도(lat)', '경도(lon)']].mean().reset_index()
mean_locations.head()

import pandas as pd
import folium
from folium import plugins

# 제주도 중심 좌표로 맵 생성 및 높은 점수 데이터 필터링
jeju_map = folium.Map(location=[33.35, 126.54], zoom_start=10)
high_score_data = jeju_grid[jeju_grid['score'] >= 0.70]

jeju_map = folium.Map(
    location=[33.35, 126.54],
    zoom_start=10,
    tiles='cartodbpositron',
    control_scale=True
)

heat_data = [[row['위도'], row['경도'], row['score']] for _, row in high_score_data.iterrows()]
plugins.HeatMap(
    heat_data,
    radius=3,
    blur=5,
    max_zoom=13,
    gradient={0.4: 'blue', 0.65: 'lime', 0.95: 'red'},
    name='잠재력 히트맵'
).add_to(jeju_map)

for idx, row in mean_locations.iterrows():
    folium.Marker(
        location=[row['위도(lat)'], row['경도(lon)']],
        popup=folium.Popup(
            folium.Html(f"""
                <div style='font-family: Arial, sans-serif; 
                            font-weight: bold; 
                            font-size: 12px; 
                            padding: 5px'>
                    {row['단지명']}
                </div>
            """, script=True),
            max_width=200
        ),
        icon=folium.Icon(
            color='black',
            icon='bolt',
            prefix='fa'
        )
    ).add_to(jeju_map)

minimap = plugins.MiniMap(toggle_display=True)
# 지도에 최소화 가능한 미니맵 추가
jeju_map.add_child(minimap)

plugins.Fullscreen().add_to(jeju_map)

folium.LayerControl(collapsed=False).add_to(jeju_map)

jeju_map.save("./output/jeju_wind_turbine_map.html")
print("저장: output/jeju_wind_turbine_map.html")
display(jeju_map)

저장: output/jeju_wind_turbine_map.html
